# Train the full model set on Colab — 5 component specialists + 6 condition specialists

**Architecture (12 models): `pole` (reused) + 5 component + 6 condition.**

| group | subset | classes | model |
|---|---|---|---|
| component | comp_wire | wire | yolo26x |
| component | comp_insulator | h_insulator, v_insulator | yolo26x |
| component | comp_crossarm | crossarm_stright, top_crossarm, om_crossarm | yolo26x |
| component | comp_vegetation | vegetation | yolo26m |
| component | comp_rust | rust | yolo26m |
| condition | cond_v_insulator | normal/band/broken/chip_off | yolo26m |
| condition | cond_h_insulator | normal/broken/chip_off | yolo26m |
| condition | cond_straight_crossarm | normal/band | yolo26m |
| condition | cond_top_crossarm | normal/band | yolo26m |
| condition | cond_om_crossarm | normal/band | yolo26m |
| condition | cond_wire | wire_normal/cross_wire | yolo26m |

Data-rich detectors (wire/insulator/crossarm) -> **yolo26x**; small/data-limited (vegetation, rust,
all condition families) -> **yolo26m** (resists overfit). `pole` is NOT retrained.
**Runtime -> A100 -> Run all.** Upload all 11 subsets to Drive first.

In [ ]:
# @title 1) Reduce CUDA fragmentation (before torch) + GPU check
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no nvidia-smi")
assert torch.cuda.is_available(), "No CUDA GPU! Runtime -> Change runtime type -> A100."
NAME=torch.cuda.get_device_name(0); VRAM=torch.cuda.get_device_properties(0).total_memory/1e9
print(f"GPU: {NAME} | VRAM {VRAM:.0f} GB | torch {torch.__version__} CUDA {torch.version.cuda}")

In [ ]:
# @title 2) Repo + deps
REPO="/content/repo"
!git clone https://github.com/arupa444/trainDronisight.git $REPO 2>/dev/null || (cd $REPO && git pull)
%cd $REPO
!pip -q install uv && uv pip install --system -e .
import torch; print("CUDA:", torch.cuda.is_available())

In [ ]:
# @title 3) Mount Drive
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# @title 4) Load the 11 trainable subsets from Drive -> /content/data
import os, glob, zipfile, shutil
DRIVE_DATA = "/content/drive/MyDrive/dronisight"     # <-- adjust to where you put the DBs
SUBSETS = ["comp_wire","comp_insulator","comp_crossarm","comp_vegetation","comp_rust",
           "cond_v_insulator","cond_h_insulator","cond_straight_crossarm","cond_top_crossarm",
           "cond_om_crossarm","cond_wire"]
DEST = "/content/data/yolo_train_db"; os.makedirs(DEST, exist_ok=True)
def locate(sub):
    for c in [f"{DRIVE_DATA}/yolo_train_db/{sub}", f"{DRIVE_DATA}/{sub}",
              f"{DRIVE_DATA}/yolo_train_db/{sub}.zip", f"{DRIVE_DATA}/{sub}.zip"]:
        if os.path.exists(c): return c
    h=glob.glob(f"{DRIVE_DATA}/**/{sub}*", recursive=True); return h[0] if h else None
for sub in SUBSETS:
    dst=f"{DEST}/{sub}"
    if os.path.isdir(dst) and glob.glob(f"{dst}/images/**/*.jpg", recursive=True): print("present:",sub); continue
    src=locate(sub); assert src, f"missing {sub} under {DRIVE_DATA}"
    if src.endswith(".zip"):
        with zipfile.ZipFile(src) as z: z.extractall(DEST)
        if not os.path.isdir(dst):
            inner=glob.glob(f"{DEST}/**/{sub}", recursive=True);  shutil.move(inner[0], dst) if inner else None
    else: shutil.copytree(src, dst, dirs_exist_ok=True)
    print("loaded:", sub)
os.environ["DRONISIGHT_DATA"]="/content/data"
import importlib, shared.config as C; importlib.reload(C)
for sub in SUBSETS:
    n=len(glob.glob(f"{C.YOLO_DB/sub}/images/train/clahe/*.jpg")); print(f"{sub}: train(clahe)={n}")
    for c in glob.glob(f"{C.YOLO_DB/sub}/**/*.cache", recursive=True): os.remove(c)

In [ ]:
# @title 5) Train all 11 (data-rich -> yolo26x, small -> yolo26m). Saves to Drive after EACH.
from train_yolo.train_components import run
from notebooks.colab_utils import save_runs_to_drive
PLAN = [   # (subset, model, imgsz, epochs)
    ("comp_wire",              "yolo26x.pt", 1280, 150),
    ("comp_insulator",         "yolo26x.pt", 1280, 150),
    ("comp_crossarm",          "yolo26x.pt", 1280, 150),
    ("comp_vegetation",        "yolo26m.pt", 1280, 150),
    ("comp_rust",              "yolo26m.pt", 1280, 200),   # data-limited -> a few more epochs
    ("cond_v_insulator",       "yolo26m.pt", 1280, 150),
    ("cond_h_insulator",       "yolo26m.pt", 1280, 150),
    ("cond_straight_crossarm", "yolo26m.pt", 1280, 150),
    ("cond_top_crossarm",      "yolo26m.pt", 1280, 150),
    ("cond_om_crossarm",       "yolo26m.pt", 1280, 150),
    ("cond_wire",              "yolo26m.pt", 1280, 150),
]
def pick_batch(model, vram):
    heavy = "26x" in model
    table = [(78,6),(40,3),(22,2),(0,1)] if heavy else [(78,32),(40,16),(22,8),(0,4)]
    for thr,b in table:
        if vram>=thr: return b
    return 1
for sub, model, imgsz, epochs in PLAN:
    batch = pick_batch(model, VRAM)
    print("="*70, f"\nTRAIN {sub}  model={model} imgsz={imgsz} batch={batch} epochs={epochs}\n", "="*70)
    run(subset=sub, version="clahe", epochs=epochs, imgsz=imgsz, batch=batch, model=model)
    print("saved to Drive:", save_runs_to_drive())

In [ ]:
# @title 6) Per-model val mAP
import glob, os
from train_yolo.eval_yolo import evaluate
for sub in ["comp_wire","comp_insulator","comp_crossarm","comp_vegetation","comp_rust",
            "cond_v_insulator","cond_h_insulator","cond_straight_crossarm","cond_top_crossarm",
            "cond_om_crossarm","cond_wire"]:
    w=sorted(glob.glob(f"runs/**/{sub}/**/weights/best.pt", recursive=True), key=os.path.getmtime)
    if w: print(evaluate(w[-1], sub, split="val", imgsz=1280), "\n")